In [5]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
#from langchain.memory import ConversationBufferWindowMemory
#from langchain_community.memory import ConversationBufferWindowMemory
from langchain_community.memory.buffer_window import ConversationBufferWindowMemory


from langchain.chains import ConversationChain
from langchain.prompts import PromptTemplate

# ----------------------------
# 1. LOAD LLM
# ----------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# ----------------------------
# 2. MEMORY (100 Q&A BUFFER)
# ----------------------------
memory = ConversationBufferWindowMemory(
    k=200,   # 100 Q + 100 A
    return_messages=True
)

def preload_memory(qa_pairs):
    """
    Load Q&A into memory
    """
    for q, a in qa_pairs:
        memory.save_context({"input": q}, {"output": a})

def load_memory(d1_df):
    for i1 in list(d1_df.index_values):
        memory.save_context({"input": d1_df.loc[i1,"Question"]}, {"output": d1_df.loc[i1,"Answer"]})

# Example dataset
qa_dataset = [
    ("What is LangChain?", "LangChain is a framework for building LLM applications."),
    ("What is cosine similarity?", "It measures semantic similarity between vectors."),
    ("Who created Python?", "Python was created by Guido van Rossum."),
]

# Simulate 100 entries
qa_dataset = qa_dataset * 34
import pandas as pd
data_df = pd.read_csv(r"C:\Users\surya.adatravu\Documents\CONV_WINDOW_LLM_ANALYSIS\RA_FSM_QA.csv")
#preload_memory(qa_dataset[:100])
load_memory(data_df)


# ----------------------------
# 3. STRICT MEMORY PROMPT
# ----------------------------
template = """
You are a strict knowledge assistant.

You must ONLY answer from conversation history.
If answer not present in history say:
"I don't know from provided knowledge."

Conversation History:
{history}

Question:
{input}

Answer strictly from history:
"""

prompt = PromptTemplate(
    input_variables=["history", "input"],
    template=template
)

chain = ConversationChain(
    llm=llm,
    memory=memory,
    prompt=prompt,
    verbose=False
)


# ----------------------------
# 4. COSINE SIMILARITY CHECKER
# ----------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def similarity_score(text1, text2):
    vec1 = embeddings.embed_query(text1)
    vec2 = embeddings.embed_query(text2)
    score = cosine_similarity([vec1], [vec2])[0][0]
    return float(score)


# ----------------------------
# 5. VALIDATE ANSWER FROM MEMORY
# ----------------------------
def validate_answer(question, llm_answer, qa_pairs, threshold=0.82):
    """
    Check if LLM answer matches any stored answer
    """
    best_score = 0
    best_answer = None

    for q, a in qa_pairs:
        sim = similarity_score(llm_answer, a)
        if sim > best_score:
            best_score = sim
            best_answer = a

    is_valid = best_score >= threshold

    return {
        "valid": is_valid,
        "score": best_score,
        "matched_answer": best_answer
    }


# ----------------------------
# 6. ASK QUESTION PIPELINE
# ----------------------------
def ask(question):
    llm_answer = chain.predict(input=question)

    validation = validate_answer(question, llm_answer, qa_dataset[:100])

    if not validation["valid"]:
        return {
            "question": question,
            "llm_answer": "Rejected: hallucination detected",
            "confidence": validation["score"]
        }

    return {
        "question": question,
        "llm_answer": llm_answer,
        "confidence": validation["score"]
    }


# ----------------------------
# 7. TEST
# ----------------------------
print(ask("What is RAG?"))
print(ask("What is hallucination?"))


ModuleNotFoundError: No module named 'langchain_community.memory.buffer_window'

In [2]:
pip show langchain

Name: langchain
Version: 1.0.3
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\surya.adatravu\AppData\Local\anaconda3\envs\r1\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip show langchain_community

Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: C:\Users\surya.adatravu\AppData\Local\anaconda3\envs\r1\Lib\site-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
Note: you may need to restart the kernel to use updated packages.
